# JARVIS_SANGRON — Hito 4.1
## K-Means: segmentación exploratoria de perfiles estudiantiles (OULAD)

**Universidad Franz Tamayo (UNIFRANZ) · Minería de Datos · 2026**

**Integrantes:** Carla Valeria Encinas Cano, Víctor Asturizaga Plata, Joel Herrera Pericon, Joel Eduardo Condoreno Chura y Vicente Oscar Claros Mamani.

**Objetivo de negocio:** complementar el sistema de alerta temprana de abandono estudiantil mediante la identificación de **perfiles académicos similares** para analizar necesidades de apoyo. **El objetivo principal sigue siendo la predicción supervisada** mediante regresión logística y árboles de decisión.

> **Importante.** K-Means es no supervisado: la variable `abandono` no se utiliza ni para seleccionar K ni para entrenar. Las tasas de abandono se calculan **después**, para describir los grupos, no para afirmar que K-Means predice riesgo o que la segmentación causa abandono.

**Material de referencia:** [clase del docente `h4_1_kmeans_ifood.ipynb`](https://github.com/ealaurel/MINERIA_DATOS_2026_2/blob/main/h4_1_kmeans_ifood.ipynb) · [repositorio JARVIS_SANGRON](https://github.com/vicentesystemdev/JARVIS_SANGRON).

### Plan de práctica (CRISP-DM)

1. **Comprensión del negocio**: identificar perfiles académicos para orientar tutorías.
2. **Comprensión de los datos**: verificar `studentInfo.csv` y su unidad inscripción–módulo–presentación.
3. **Preparación**: seleccionar características disponibles antes de conocer el resultado, imputar, codificar y escalar **solo usando entrenamiento**.
4. **Modelado no supervisado**: contrastar K=2 a K=6 mediante WCSS e índice Silhouette.
5. **Evaluación**: interpretar los grupos en registros de prueba no utilizados en el ajuste, con sus tasas de abandono históricas y límites.
6. **Recomendación**: proponer apoyos académicos, sin asignar automáticamente etiquetas de riesgo a individuos.

**Diseño de variables**: `studied_credits`, `num_of_prev_attempts` y `highest_education`. Son dos medidas cuantitativas de trayectoria académica y una categoría de educación previa. La codificamos con *one-hot* para no inventar distancias numéricas entre títulos.

**Exclusiones**: `final_result` y `abandono` por fuga de objetivo, `id_student` por ser identificador; además `gender`, `age_band`, `disability`, `region` e `imd_band` no forman clusters para evitar usar directamente atributos sensibles o contextuales como criterio de agrupación. `code_module` y `code_presentation` se estudian posteriormente como contexto.

**Advertencia metodológica:** K-Means usa distancia euclidiana sobre una mezcla de columnas escaladas y one-hot; el resultado depende de las variables/pesos y no equivale a clusters naturales probados.

In [ ]:
from pathlib import Path
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 777  # Coherente con los notebooks de Hito 3
NUMERICAS = ['studied_credits', 'num_of_prev_attempts']
CATEGORICAS = ['highest_education']
VARIABLES = NUMERICAS + CATEGORICAS
K_CANDIDATOS = list(range(2, 7))
TAM_MUESTRA_SILHOUETTE = 2000  # evita el coste cuadrático de usar todas las filas
FUENTE = 'Fuente: OULAD (studentInfo.csv). Elaboración propia.'
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 100})
pd.set_option('display.max_columns', 30)
print('Librerías cargadas. Semilla:', SEMILLA)

## 1. Dataset y calidad inicial

El archivo original incluye aproximadamente 32.593 inscripciones y 28.785 estudiantes distintos (un mismo `id_student` puede aparecer en más de una inscripción). No utilizamos el conjunto iFood: este es el dataset real de nuestro proyecto.

El notebook funciona desde la **raíz del repositorio** o dentro de `Codigo/`, buscando `Data/studentInfo.csv`. Si descargaste el notebook aislado, ubícalo en `Codigo/` y mantén `Data/` en la raíz.

In [ ]:
rutas = [Path('Data/studentInfo.csv'), Path('../Data/studentInfo.csv')]
ruta_csv = next((p.resolve() for p in rutas if p.exists()), None)
if ruta_csv is None:
    raise FileNotFoundError('Falta Data/studentInfo.csv. Ejecuta el notebook dentro del repositorio JARVIS_SANGRON.')

raw = pd.read_csv(ruta_csv)
columnas_obligatorias = set(VARIABLES + ['id_student','final_result','code_module','code_presentation'])
assert columnas_obligatorias.issubset(raw.columns), f'Faltan columnas: {columnas_obligatorias - set(raw.columns)}'
assert raw['id_student'].notna().all(), 'Hay estudiantes sin identificador.'
assert raw['final_result'].notna().all(), 'El resultado final tiene valores nulos.'
raw['abandono'] = raw['final_result'].eq('Withdrawn').astype(int)
assert not {'final_result', 'abandono', 'id_student'}.intersection(VARIABLES)
print('Archivo:', ruta_csv)
print('SHA256:', hashlib.sha256(ruta_csv.read_bytes()).hexdigest())
print('Inscripciones:', len(raw), '| Estudiantes:', raw['id_student'].nunique())
print('Tasa histórica de abandono: {:.2%}'.format(raw['abandono'].mean()))
display(raw[VARIABLES + ['code_module','final_result']].head())

In [ ]:
display(pd.DataFrame({
    'Tipo': raw[VARIABLES].dtypes.astype(str),
    'Nulos': raw[VARIABLES].isna().sum(),
    'Porcentaje nulos': (100*raw[VARIABLES].isna().mean()).round(2),
}))
display(raw[NUMERICAS].describe().round(2))
print('Categorías educación:', raw['highest_education'].value_counts(dropna=False).to_dict())

## 2. Partición por estudiante y preparación

Reproducimos la **misma lógica y semilla** de las entregas del Hito 3: 70 % de estudiantes para entrenamiento y 30 % restantes divididos para validación/prueba. La proporción de filas puede diferir ligeramente de 70/15/15 porque cada estudiante puede tener varias inscripciones. Ninguna persona debe estar en más de un conjunto.

Toda imputación, codificación y escalado se aprende exclusivamente en entrenamiento. Esto evita incluir información de la muestra de prueba en el cálculo de los centroides.

In [ ]:
indices = np.arange(len(raw))
grupos = raw['id_student'].to_numpy()
idx_train, idx_temp = next(GroupShuffleSplit(n_splits=1, test_size=.30,
                                            random_state=SEMILLA).split(indices, groups=grupos))
idx_val_rel, idx_test_rel = next(GroupShuffleSplit(n_splits=1, test_size=.50,
                                                   random_state=SEMILLA).split(idx_temp, groups=grupos[idx_temp]))
partes = {'Entrenamiento': idx_train,
          'Validación': idx_temp[idx_val_rel],
          'Prueba': idx_temp[idx_test_rel]}
ids = [set(raw.iloc[p]['id_student']) for p in partes.values()]
assert all(ids[i].isdisjoint(ids[j]) for i in range(3) for j in range(i+1, 3)), 'Fuga de estudiantes entre conjuntos.'
assert len(np.unique(np.concatenate(list(partes.values())))) == len(raw)
resumen_partes = pd.DataFrame([
    {'Conjunto': nombre, 'Inscripciones': len(p), 'Estudiantes únicos': raw.iloc[p]['id_student'].nunique()}
    for nombre, p in partes.items()
])
display(resumen_partes)
print('Partición sin solapamiento de id_student: OK')

In [ ]:
preparacion = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputar', SimpleImputer(strategy='median')),
                      ('escalar', StandardScaler())]), NUMERICAS),
    ('cat', Pipeline([('imputar', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAS)
], remainder='drop', sparse_threshold=0)

X_train = preparacion.fit_transform(raw.iloc[partes['Entrenamiento']][VARIABLES])
X_val = preparacion.transform(raw.iloc[partes['Validación']][VARIABLES])
X_test = preparacion.transform(raw.iloc[partes['Prueba']][VARIABLES])
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
print('Matrices preparadas:', 'TRAIN', X_train.shape, '| VAL', X_val.shape, '| TEST', X_test.shape)
print('Variables codificadas:', list(preparacion.get_feature_names_out()))

## 3. ¿Cuántos clusters? WCSS (Codo) y Silhouette

K-Means minimiza la suma de distancias cuadráticas al centroide más cercano, que scikit-learn denomina **inercia** o **WCSS**. La inercia disminuye al aumentar K, así que **no debe seleccionarse K únicamente por el mínimo WCSS**.

El índice Silhouette oscila entre -1 y 1: valores más altos sugieren mayor separación y cohesión. Medimos Silhouette en una muestra reproducible de entrenamiento para limitar su coste computacional. Comparamos K=2…6; ninguna etiqueta de abandono interviene en esta decisión.

In [ ]:
resultados_k = []
for k in K_CANDIDATOS:
    modelo_tmp = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10)
    etiquetas_train = modelo_tmp.fit_predict(X_train)
    s = silhouette_score(X_train, etiquetas_train,
                         sample_size=min(TAM_MUESTRA_SILHOUETTE, len(X_train)),
                         random_state=SEMILLA)
    resultados_k.append({'K': k, 'WCSS': modelo_tmp.inertia_, 'Silhouette_muestra': s})

tabla_k = pd.DataFrame(resultados_k)
K_MEJOR = int(tabla_k.loc[tabla_k['Silhouette_muestra'].idxmax(), 'K'])
display(tabla_k.round(4))
print('K mejor por Silhouette (muestra de TRAIN):', K_MEJOR)
print('K=4 se usa en el ejemplo de iFood, pero NO se impone aquí.')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(tabla_k['K'], tabla_k['WCSS'], 'o-')
ax[0].set(title='Método del Codo — OULAD', xlabel='Número de clusters (K)', ylabel='WCSS / Inercia')
ax[1].plot(tabla_k['K'], tabla_k['Silhouette_muestra'], 'o-')
ax[1].axvline(K_MEJOR, linestyle='--', label=f'Mejor Silhouette: K={K_MEJOR}')
ax[1].set(title='Silhouette — muestra de entrenamiento', xlabel='Número de clusters (K)', ylabel='Silhouette')
ax[1].legend()
for a in ax: a.grid(alpha=.25)
plt.tight_layout()
plt.show()
print(FUENTE)

## 4. Entrenamiento final y segmentación de inscripciones de prueba

El modelo se **ajusta solo con TRAIN**. Después asignamos un cluster a cada inscripción de TEST mediante `predict`, sin volver a entrenar ni modificar los centroides. No usamos `abandono` para construir el algoritmo.

Las etiquetas 0, 1, etc., son identificadores arbitrarios, **no niveles de riesgo**.

In [ ]:
modelo = KMeans(n_clusters=K_MEJOR, random_state=SEMILLA, n_init=10)
modelo.fit(X_train)
cluster_test = modelo.predict(X_test)
cluster_train = modelo.labels_
print('K seleccionado:', K_MEJOR)
print('WCSS entrenamiento:', round(modelo.inertia_, 2))
print('Clientes/inscripciones por cluster en TEST:')
print(pd.Series(cluster_test, name='Cluster').value_counts().sort_index())

## 5. Perfilado descriptivo (sin fuga de etiquetas)

El porcentaje de `Withdrawn` se calcula **solo ahora**, al describir los clusters sobre el conjunto TEST. Sirve para discutir diferencias históricas entre perfiles; no convierte a K-Means en un clasificador ni valida intervenciones futuras. Además, comparar grupos sin pruebas de incertidumbre no demuestra diferencias causales.

In [ ]:
perfil_test = raw.iloc[partes['Prueba']].copy()
perfil_test['Cluster'] = cluster_test
resumen = (perfil_test.groupby('Cluster')
    .agg(Inscripciones=('id_student','size'),
         Estudiantes=('id_student','nunique'),
         Creditos_medios=('studied_credits','mean'),
         Intentos_previos_medios=('num_of_prev_attempts','mean'),
         Abandonos=('abandono','sum'),
         Tasa_abandono=('abandono','mean'))
    .reset_index())
resumen['Participacion_pct'] = 100*resumen['Inscripciones']/len(perfil_test)
resumen['Abandono_pct'] = 100*resumen['Tasa_abandono']
print('Tasa general en TEST: {:.2%}'.format(perfil_test['abandono'].mean()))
display(resumen[['Cluster','Inscripciones','Estudiantes','Participacion_pct',
                 'Creditos_medios','Intentos_previos_medios','Abandonos','Abandono_pct']].round(2))
print('Educación previa predominante por cluster (TEST):')
display(pd.crosstab(perfil_test['Cluster'], perfil_test['highest_education'], normalize='index').mul(100).round(1))
print('Módulos por cluster (TEST), % dentro del cluster:')
display(pd.crosstab(perfil_test['Cluster'], perfil_test['code_module'], normalize='index').mul(100).round(1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(resumen['Cluster'].astype(str), resumen['Inscripciones'])
ax[0].set(title='Tamaño de los perfiles (TEST)', xlabel='Cluster', ylabel='Inscripciones')
ax[1].bar(resumen['Cluster'].astype(str), resumen['Abandono_pct'])
ax[1].axhline(perfil_test['abandono'].mean()*100, linestyle='--',
              label='Tasa total TEST')
ax[1].set(title='Abandono histórico por perfil (TEST)', xlabel='Cluster', ylabel='Abandono (%)')
ax[1].legend()
plt.tight_layout();plt.show()
print(FUENTE)

## 6. Visualización 2D con PCA

PCA proyecta para visualizar en dos dimensiones el espacio de características **ya transformado**. Se ajusta solo en TRAIN y se aplica en TEST. Las distancias del modelo K-Means **no se recalculan** sobre el dibujo; una proyección 2D puede ocultar separaciones.

In [ ]:
pca = PCA(n_components=2, random_state=SEMILLA)
pca.fit(X_train)
xy = pca.transform(X_test)
plt.figure(figsize=(9,6))
sc = plt.scatter(xy[:,0], xy[:,1], c=cluster_test, cmap='tab10',
                 alpha=.45, s=13, linewidths=0)
plt.colorbar(sc, ticks=np.arange(K_MEJOR), label='Cluster')
plt.title('PCA de los perfiles académicos (TEST)')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.tight_layout();plt.show()
print('Varianza explicada por 2 componentes: {:.1%}'.format(pca.explained_variance_ratio_.sum()))
print(FUENTE)

## 7. Interpretación y decisiones de negocio

Las decisiones deben basarse en **los resultados calculados**, no en una correspondencia preasignada entre el número de cluster y un supuesto nivel de riesgo.

- Un perfil con créditos inusuales puede necesitar revisión de carga académica.
- Un perfil con más intentos previos puede motivar apoyo en asignaturas críticas.
- La educación previa puede orientar recursos de nivelación, **sin negar servicios por categoría**.
- Una tasa de abandono elevada dentro de un cluster es **una asociación retrospectiva**, no una probabilidad de abandono validada para cada estudiante.
- Ningún modelo se utilizará para etiquetar automáticamente personas ni para decisiones adversas. Se recomiendan apoyos voluntarios y evaluación posterior de las intervenciones.

La fase CRISP-DM de despliegue y la reducción de abandono del 10 % siguen siendo **metas no demostradas**.

In [ ]:
display(Markdown(f"""
### Resumen reproducible de la práctica
- **Dataset:** OULAD `studentInfo.csv` ({len(raw):,} inscripciones, {raw.id_student.nunique():,} estudiantes).
- **Modelo:** K-Means, K={K_MEJOR}, entrenado sobre {len(X_train):,} inscripciones.
- **Variables:** {', '.join(VARIABLES)}; transformación ajustada solo en TRAIN.
- **Silhouette (muestra TRAIN):** {tabla_k.set_index('K').loc[K_MEJOR,'Silhouette_muestra']:.4f}.
- **Inercia TRAIN:** {modelo.inertia_:,.2f}.
- **TEST:** {len(perfil_test):,} inscripciones; abandono histórico global {perfil_test.abandono.mean():.2%}.
- **Control:** la etiqueta `abandono` se usa solo para perfilar clusters después del ajuste.
- **Límite:** es segmentación exploratoria; no reemplaza la predicción supervisada ni mide reducción real del abandono.
"""))

## 8. Preguntas para la defensa / actividad del docente

1. ¿Por qué K-Means es aprendizaje no supervisado y la regresión logística es supervisada?
2. ¿Qué significa un centroide en el espacio de dos variables numéricas y educación codificada?
3. ¿Por qué usamos `StandardScaler` y `OneHotEncoder` en vez de codificar la educación como 0, 1, 2…?
4. ¿Con qué criterio seleccionamos K y qué limitación tiene el índice Silhouette?
5. ¿Qué grupos podrían beneficiarse de tutorías o nivelación y cuál es la evidencia observada?
6. ¿Por qué `final_result`, `abandono` e `id_student` no pueden formar parte del agrupamiento?
7. ¿Por qué una tasa de abandono superior dentro de un cluster no implica que el cluster prediga abandono individual ni que sus características lo causen?

### Referencias
- Docente E. Laurel, `h4_1_kmeans_ifood.ipynb` (2026).
- JARVIS_SANGRON, entregas H3 y `README.md`.
- Kuzilek, Hlosta & Zdrahal (2017), *Open University Learning Analytics dataset*, Scientific Data.
- [Documentación scikit-learn: KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

**Nota de entrega:** este notebook es complementario. No modificar las entregas validadas del Hito 3. Ejecutar **Restart Kernel and Run All Cells** antes de entregar y verificar los resultados.

## 9. Respuestas para la actividad / defensa del Hito 4.1

**1. ¿Por qué K-Means es aprendizaje no supervisado y la regresión logística es supervisada?**

K-Means es un método no supervisado porque forma grupos de inscripciones según la similitud de sus características, sin utilizar el resultado académico conocido. La regresión logística es supervisada porque aprende a partir de la variable objetivo `abandono` para estimar una probabilidad de retiro. En JARVIS_SANGRON ambos métodos son complementarios: uno segmenta perfiles y el otro predice.

**2. ¿Qué significa un centroide en el espacio de dos variables numéricas y educación codificada?**

El centroide es el promedio de las coordenadas transformadas de las inscripciones de un cluster. Sus dos componentes numéricos describen créditos e intentos previos en escala estandarizada; las componentes *one-hot* de `highest_education` expresan proporciones de cada nivel educativo en el grupo. No necesariamente corresponde a un estudiante real.

**3. ¿Por qué usamos `StandardScaler` y `OneHotEncoder` en vez de codificar la educación como 0, 1, 2…?**

Como K-Means usa distancias, `StandardScaler` evita que los créditos dominen simplemente por tener magnitudes mayores que los intentos previos. `OneHotEncoder` representa cada categoría educativa con una columna binaria y evita imponer separaciones numéricas arbitrarias entre niveles educativos. Las categorías codificadas siguen influyendo en las distancias, por lo que la selección y ponderación de variables deben justificarse.

**4. ¿Con qué criterio seleccionamos K y qué limitación tiene el índice Silhouette?**

Comparamos K=2…6 usando WCSS (Método del Codo) y el índice Silhouette. En esta práctica elegimos el K con mayor Silhouette calculado sobre una muestra reproducible del entrenamiento y contrastamos el resultado con la curva del codo y su interpretabilidad. Silhouette favorece grupos compactos y bien separados, pero no demuestra que sean estables, causalmente relevantes o útiles para orientar acciones educativas. El **K y el valor Silhouette específicos** aparecen en las celdas ejecutadas de la sección 3.

**5. ¿Qué grupos podrían beneficiarse de tutorías o nivelación y cuál es la evidencia observada?**

Podrían considerarse para apoyo los grupos con mayor promedio de intentos previos, cargas académicas que requieran seguimiento o necesidades potenciales de nivelación. La evidencia debe basarse en el perfil estadístico de cada cluster, su composición educativa y su tasa histórica de `abandono` en TEST, comparada con la tasa general. **No se asigna un número de cluster a priori**: la tabla calculada en la sección 5 identifica qué perfiles muestran esas características. Las asociaciones justifican estudiar apoyos voluntarios, no imponer decisiones individuales.

**6. ¿Por qué `final_result`, `abandono` e `id_student` no pueden formar parte del agrupamiento?**

`final_result` y su derivada `abandono` contienen el desenlace que queremos estudiar después de la segmentación; incluirlas crearía fuga de información y produciría grupos condicionados por el resultado. `id_student` solo es un identificador, sin significado académico como distancia. Lo utilizamos para separar inscripciones del mismo estudiante entre entrenamiento, validación y prueba, pero no como predictor o variable de agrupamiento.

**7. ¿Por qué una tasa de abandono superior dentro de un cluster no implica que el cluster prediga abandono individual ni que sus características lo causen?**

Una tasa por cluster es una frecuencia histórica agregada: no garantiza el resultado de cada estudiante ni constituye una predicción individual validada. Además, la asociación puede estar influida por variables no medidas y no demuestra que los créditos, intentos previos o nivel educativo causen abandono. Para predicciones individuales usamos modelos supervisados evaluados fuera de entrenamiento; para atribuir efectos a una tutoría necesitaríamos evaluar una intervención.

> **Nota académica:** las respuestas 4 y 5 se apoyan en las tablas y gráficos calculados al ejecutar el notebook con `Data/studentInfo.csv`. No se inventan resultados numéricos ni etiquetas de clusters antes de su ejecución.
